# 02_Practice: 토크나이저 직접 비교

## 목표
BPE, WordPiece, SentencePiece를 실제로 로드하고 비교

## 1. 라이브러리 설치 및 로드

In [ ]:
# pip install transformers pandas matplotlib numpy

from transformers import AutoTokenizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("라이브러리 로드 완료")

## 2. 토크나이저 로드

In [ ]:
print("토크나이저 로드 중...\n")

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
xlm_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

print(f"✓ GPT-2 (BPE): {len(gpt2_tokenizer):,}개 어휘")
print(f"✓ mBERT (WordPiece): {len(mbert_tokenizer):,}개 어휘")
print(f"✓ XLM (SentencePiece): {len(xlm_tokenizer):,}개 어휘")

## 3. 테스트 텍스트

In [ ]:
test_cases = {
    "English": "I love machine learning and natural language processing.",
    "Korean": "기계 학습과 자연어 처리를 좋아합니다.",
    "Mixed": "I love 자연어 processing!"
}

for lang, text in test_cases.items():
    print(f"[{lang}] {text}")

## 4. 토크나이징 결과

In [ ]:
# 각 토크나이저로 처리
results = {}

for lang, text in test_cases.items():
    gpt2_ids = gpt2_tokenizer.encode(text)
    mbert_ids = mbert_tokenizer.encode(text)
    xlm_ids = xlm_tokenizer.encode(text)
    
    results[lang] = {
        'GPT-2': len(gpt2_ids),
        'mBERT': len(mbert_ids) - 2,  # [CLS], [SEP] 제외
        'XLM': len(xlm_ids) - 2  # <s>, </s> 제외
    }

# 결과 표시
df = pd.DataFrame(results).T
print("\n토큰 개수 비교:\n")
print(df)

## 5. 차이점 분석

In [ ]:
print("\n차이점 분석:\n")

for lang in results.keys():
    gpt2 = results[lang]['GPT-2']
    mbert = results[lang]['mBERT']
    xlm = results[lang]['XLM']
    
    print(f"[{lang}]")
    print(f"  GPT-2 vs mBERT: {gpt2-mbert:+d}개")
    print(f"  GPT-2 vs XLM:   {gpt2-xlm:+d}개")
    
    best = min(gpt2, mbert, xlm)
    if best == gpt2:
        print(f"  → 최효율: GPT-2")
    elif best == mbert:
        print(f"  → 최효율: mBERT")
    else:
        print(f"  → 최효율: XLM")
    print()

## 6. 비용 계산 (OpenAI GPT-4 기준: 1000토큰당 $0.03)

In [ ]:
cost_per_1k = 0.03

print("\nAPI 비용 영향:\n")

for lang in results.keys():
    gpt2 = results[lang]['GPT-2']
    mbert = results[lang]['mBERT']
    xlm = results[lang]['XLM']
    
    gpt2_cost = gpt2 * cost_per_1k / 1000
    mbert_cost = mbert * cost_per_1k / 1000
    xlm_cost = xlm * cost_per_1k / 1000
    
    print(f"[{lang}]")
    print(f"  GPT-2:  ${gpt2_cost:.6f}")
    print(f"  mBERT:  ${mbert_cost:.6f} ({(1-mbert_cost/gpt2_cost)*100:.0f}% 절감)")
    print(f"  XLM:    ${xlm_cost:.6f} ({(1-xlm_cost/gpt2_cost)*100:.0f}% 절감)")
    print()

## 7. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df))
width = 0.25

ax.bar(x - width, df['GPT-2'], width, label='GPT-2', color='#FF6B6B')
ax.bar(x, df['mBERT'], width, label='mBERT', color='#4ECDC4')
ax.bar(x + width, df['XLM'], width, label='XLM', color='#45B7D1')

ax.set_ylabel('Token Count', fontsize=12)
ax.set_title('Token Count Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df.index)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 결론

### 언어별 최적 토크나이저

| 언어 | 최적 | 이유 |
|------|------|------|
| 영어 | BPE | 효율성 우수 |
| 한글 | WordPiece/SentencePiece | 한글 처리 최적화 |
| 혼합 | SentencePiece | 다국어 일관성 |

### 선택 가이드
- **비용 중심**: SentencePiece
- **의미 중심**: WordPiece
- **다국어**: SentencePiece (필수)